# Demo 15 — Dalla STFT di nuovo all'audio

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
y_ritorno, sr_ritorno = al.load_audio(AUDIO / "C-major-scale.wav", sr=SR)
y_ritorno = y_ritorno.astype(np.float64)
D_ritorno = al.analysis.stft(y_ritorno, n_fft=1024, win_length=1024, hop_length=256,
                         window="hann", center=True)
y_ricostruito = al.analysis.istft(D_ritorno, win_length=1024, hop_length=256, window="hann",
                             center=True, length=len(y_ritorno))
y_fase_zero = al.analysis.istft(abs(D_ritorno).astype(complex), win_length=1024, hop_length=256,
                           window="hann", center=True, length=len(y_ritorno))
np.testing.assert_allclose(y_ricostruito, y_ritorno, atol=1e-10)
print("Errore massimo con STFT complessa:", np.max(abs(y_ricostruito-y_ritorno)))
print("Errore RMS con fase zero:", np.sqrt(np.mean((y_fase_zero-y_ritorno)**2)))
print("Frame centrati:", D_ritorno.shape[1], "attesi:", 1+len(y_ritorno)//256)
assert D_ritorno.shape[1] == 1+len(y_ritorno)//256
D_non_centrata = al.analysis.stft(y_ritorno, n_fft=1024, hop_length=256, center=False)
assert D_non_centrata.shape[1] == 1+(len(y_ritorno)-1024)//256
print("Frame senza padding:", D_non_centrata.shape[1])

inizio = min(int(0.5 * sr_ritorno), len(y_ritorno) // 2)
sezione = slice(inizio, inizio + int(0.04 * sr_ritorno))
t_ritorno = np.arange(len(y_ritorno)) / sr_ritorno

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Ricostruzione fedele: coincide", "Fase azzerata: cambia"))
fig.add_trace(go.Scatter(x=t_ritorno[sezione], y=y_ritorno[sezione], mode="lines",
                         line=dict(width=1.2, color="#1769C2"), name="originale"), row=1, col=1)
fig.add_trace(go.Scatter(x=t_ritorno[sezione], y=y_ricostruito[sezione], mode="lines",
                         line=dict(width=1.2, color="#D47426", dash="dash"),
                         name="ricostruito"), row=1, col=1)
fig.add_trace(go.Scatter(x=t_ritorno[sezione], y=y_ritorno[sezione], mode="lines",
                         line=dict(width=1.2, color="#1769C2"), name="originale", showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=t_ritorno[sezione], y=y_fase_zero[sezione], mode="lines",
                         line=dict(width=1.2, color="#2E8B57"), name="fase azzerata"), row=2, col=1)
fig.update_xaxes(title_text="tempo [s]", row=2, col=1)
fig.update_layout(height=480, title="Dalla STFT di nuovo all'audio")
fig.show()
# Guadagno comune alle tre copie per l'ascolto, senza normalizzazioni separate.
guadagno_ritorno = min(1.0, 0.9/max(np.max(abs(x)) for x in [y_ritorno, y_ricostruito, y_fase_zero]))
for nome, x in [("originale", y_ritorno), ("ricostruito", y_ricostruito), ("fase zero", y_fase_zero)]:
    print(nome); display(al.safe_audio(guadagno_ritorno*x, sr_ritorno, normalize=False))
